# Exa Agent

The Exa Agent is an AI-powered research assistant with direct access to [Exa's](https://exa.ai) neural search engine. It combines three capabilities — general web search, code and documentation lookup, and full-text content extraction from any URL — to answer questions with up-to-date, sourced information.

In [1]:
!pip install -q aixplain requests

## Add your API keys

Get your aiXplain access key from the [Integrations](https://platform.aixplain.com/account/integrations) page.

Get your Exa API key from the [Exa Dashboard](https://dashboard.exa.ai/api-keys).

In [ ]:
AixplainAPIKey = "<YOUR_AIXPLAIN_API_KEY>"
ExaApiKey      = "<YOUR_EXA_API_KEY>"

import os
os.environ["TEAM_API_KEY"] = AixplainAPIKey
os.environ["EXA_API_KEY"]  = ExaApiKey

In [3]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)

# What is the Exa Agent?

The goal of this agent is to give an AI assistant live access to the web — both for searching new content and for reading the full text of specific pages.

## Agent Architecture

The agent uses three custom script tools, each wrapping a different Exa API endpoint:

- **Web Search** — neural web search for current information, news, and facts, with configurable search type (`auto`, `neural`, `fast`, or `deep`).
- **Code Context** — targets GitHub, Stack Overflow, and official documentation to find code examples and API references.
- **Content Extraction** — retrieves the full text of one or more URLs, useful for reading articles, documentation pages, or any web content in full.

## Agent Workflow

1. **Understand the request** — determine whether it needs a search, a code lookup, or reading a specific URL.
2. **Call the right tool** — route to web search, code context, or content extraction accordingly.
3. **Synthesise results** — summarise findings, cite sources, and present a clear answer.

In [4]:
def web_search_exa(query: str, num_results: int = 8, search_type: str = "auto") -> str:
    """
    Searches the web using Exa's neural search API for current information, news, or facts.

    Parameters:
        query (str): The search query.
        num_results (int): Number of results to return (default: 8).
        search_type (str): Search speed/depth — 'auto' (default), 'neural', 'fast', or 'deep'.

    Returns:
        str: Formatted search results with titles, URLs, and content highlights.
    """
    import requests
    import os

    api_key = os.environ.get("EXA_API_KEY")
    if not api_key:
        return "Error: EXA_API_KEY environment variable is not set."

    url = "https://api.exa.ai/search"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
    }
    payload = {
        "query": query,
        "type": search_type,
        "numResults": max(1, min(num_results, 50)),
        "contents": {
            "highlights": {
                "maxCharacters": 4000,
                "numSentences": 3,
            }
        },
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}\nResponse: {e.response.text if e.response else 'N/A'}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    results = data.get("results", [])
    if not results:
        return "No results found."

    formatted = []
    for i, item in enumerate(results, start=1):
        title = item.get("title", "No title")
        link = item.get("url", "")
        highlights = item.get("highlights", [])
        snippet = " ".join(highlights) if highlights else item.get("text", "")[:400]
        formatted.append(f"[{i}] {title}\nURL: {link}\n{snippet}")

    return "\n\n".join(formatted)

In [5]:
def get_code_context_exa(query: str, num_results: int = 5) -> str:
    """
    Searches for code examples and documentation using Exa's neural search API.

    Parameters:
        query (str): The code or API search query, e.g. 'Python async context manager example',
                     'React useEffect cleanup function'.
        num_results (int): Number of results to return (default: 5).

    Returns:
        str: Formatted results with titles, URLs, and code/documentation snippets.
    """
    import requests
    import os

    api_key = os.environ.get("EXA_API_KEY")
    if not api_key:
        return "Error: EXA_API_KEY environment variable is not set."

    url = "https://api.exa.ai/search"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
    }
    payload = {
        "query": query,
        "type": "auto",
        "numResults": max(1, min(num_results, 20)),
        "includeDomains": [
            "github.com", "stackoverflow.com", "docs.python.org",
            "developer.mozilla.org", "reactjs.org", "docs.rs",
        ],
        "contents": {
            "text": {"maxCharacters": 5000}
        },
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}\nResponse: {e.response.text if e.response else 'N/A'}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    results = data.get("results", [])
    if not results:
        return "No results found."

    formatted = []
    for i, item in enumerate(results, start=1):
        title = item.get("title", "No title")
        link = item.get("url", "")
        text = item.get("text", "")[:600]
        formatted.append(f"[{i}] {title}\nURL: {link}\n{text}")

    return "\n\n".join(formatted)

In [6]:
def extract_content_exa(urls: list, max_chars: int = 8000) -> str:
    """
    Retrieves the full text content of one or more URLs using Exa's content extraction API.

    Parameters:
        urls (list): A list of URLs to extract content from.
                     Examples: ['https://example.com/article', 'https://docs.example.com/guide']
        max_chars (int): Maximum characters to return per URL (default: 8000).

    Returns:
        str: The extracted full text from each URL, labelled by source.
    """
    import requests
    import os

    api_key = os.environ.get("EXA_API_KEY")
    if not api_key:
        return "Error: EXA_API_KEY environment variable is not set."

    if isinstance(urls, str):
        urls = [urls]

    endpoint = "https://api.exa.ai/contents"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
    }
    payload = {
        "ids": urls,
        "text": {"maxCharacters": max(500, min(max_chars, 20000))},
    }

    try:
        response = requests.post(endpoint, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}\nResponse: {e.response.text if e.response else 'N/A'}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    results = data.get("results", [])
    if not results:
        return "No content could be extracted from the provided URLs."

    formatted = []
    for item in results:
        title = item.get("title", "No title")
        url = item.get("url", "")
        text = item.get("text", "").strip()
        formatted.append(f"--- {title} ---\nURL: {url}\n\n{text}")

    return "\n\n".join(formatted)

In [8]:
SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

# Inject API key directly into each tool's source — os.environ is not
# available in aiXplain's cloud tool runtime, so the key must be embedded.
def _inject(fn, key_value):
    src = inspect.getsource(fn)
    return src.replace('os.environ.get("EXA_API_KEY")', f'"{key_value}"')

web_search_tool = aix.Tool(
    name="Exa Search",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": _inject(web_search_exa, ExaApiKey), "function_name": "web_search_exa"},
)
web_search_tool.save()
print(f"Created: {web_search_tool.name}")

code_context_tool = aix.Tool(
    name="Exa Code Context",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": _inject(get_code_context_exa, ExaApiKey), "function_name": "get_code_context_exa"},
)
code_context_tool.save()
print(f"Created: {code_context_tool.name}")

content_extraction_tool = aix.Tool(
    name="Exa Content Extraction",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": _inject(extract_content_exa, ExaApiKey), "function_name": "extract_content_exa"},
)
content_extraction_tool.save()
print(f"Created: {content_extraction_tool.name}")

Created: Exa Search
Created: Exa Code Context
Created: Exa Content Extraction


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, three tools covering search, code lookup, and full-page extraction.
* The **instructions** that guide the agent's behaviour and tool selection.

In [9]:
exa_agent = aix.Agent(
    name="Exa Agent",
    description="Neural web search agent that searches the live web, finds code examples, and extracts full content from any URL using Exa's search API.",
    instructions="""
    You are a research assistant with live access to the web via Exa's neural search API.
    You have three tools — choose the most appropriate one for each request:

    - Exa Web Search: for general questions, current news, facts, or any topic needing
      up-to-date information. Use search_type="deep" for complex research questions.

    - Exa Code Context: for programming questions, API usage, library documentation,
      or anything involving code. This targets GitHub, Stack Overflow, and official docs.

    - Exa Content Extraction: when the user provides a specific URL and wants you to
      read, summarise, or analyse that page's full content. Also use this to follow up
      on a search result that looks highly relevant.

    Guidelines:
    - Always cite sources with titles and URLs.
    - Summarise findings clearly and concisely.
    - If a search returns weak results, rephrase the query and try again.
    - For multi-part questions, call multiple tools and synthesise the results.
    - Never present information as fact without grounding it in a search result or extracted content.
    """,
    tools=[web_search_tool, code_context_tool, content_extraction_tool],
)
exa_agent.save()
print(f"Agent created: {exa_agent.id}")

Agent created: 69c460119e1dff6231a19c0c


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.

In [10]:
result = exa_agent.run("What are the latest developments in AI agents?")
print(result.data.output)

Recent developments in AI agents include:

1. MiroMind's MiroThinker-1.7 and MiroThinker-H1: These new AI systems have achieved state-of-the-art results in multi-step reasoning and deep research benchmarks, outperforming leading models like GPT-5.4 and Claude-4.6-Opus. They introduce a concept called "Effective Interaction Scaling" to improve reasoning quality and reliability. Read more here: https://www.prnewswire.com/news-releases/miromind-team-unveils-mirothinker-1-7--mirothinker-h1-a-new-era-of-verification-centric-heavy-duty-research-agents-301515678.html

2. NVIDIA's Nemotron 3 Super: This model delivers five times higher throughput for agentic AI, utilizing a hybrid mixture-of-experts architecture to enhance efficiency and accuracy. It is designed for complex tasks in multi-agent systems, significantly improving performance in various applications. Read more here: https://blogs.nvidia.com/blog/2026/03/11/nemotron-3-super/


In [11]:
result2 = exa_agent.run("Show me how to use the Exa Python SDK to search and retrieve content")
print(result2.data.output)

To use the Exa Python SDK to search and retrieve content, follow these steps:

Installation:
To install the Exa Python SDK, run the following command:
pip install exa-py
Note: Requires Python 3.9+

Quick Start:
Here’s a simple example of how to use the Exa Python SDK to perform a web search:
from exa_py import Exa

exa = Exa(api_key="your-api-key")

# Search the web
results = exa.search(
    "blog post about AI"
)
This code initializes the Exa client with your API key and performs a search for blog posts about AI. You can replace the search query with any topic of your choice.

For more detailed documentation, you can visit the Exa Python Package GitHub Repository: https://github.com/exa-labs/exa-py


In [12]:
# Content extraction: read the full text of a specific URL
result3 = exa_agent.run("Read and summarise this page: https://docs.exa.ai/reference/search")
print(result3.data.output)

The extracted content provides an overview of the Exa Search API, which allows users to perform internet-scale searches and retrieve results from various sources. It includes details on how to obtain an API key, the OpenAPI specifications, and the search endpoint for executing queries. For more information, users can refer to the complete documentation index.


In [13]:
# Multi-turn: follow up on the previous result
result4 = exa_agent.run(
    "What parameters does the highlights option support?",
    session_id=result3.data.session_id
)
print(result4.data.output)

The highlights option in the Exa Search API supports the following parameters:
1. maxCharacters: Maximum characters for highlights.
2. query: Custom query to direct highlight selection.


# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.

In [14]:
exa_agent.save()

Agent(path=asma-furniturewala/exa-agent/aixplain)

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).